# Experiment 2 — PhotonFlow on MNIST (v17 champion)

**Status**: this is the **slimmed champion** of the gap-sweep sprint
(`kaggle/photonflow-gap-sweep-2k-steps` v17, see
`photonflow_experiments_report.md` §12 for the full sweep).

**Result at 2 K steps** (eval = uniform-t plain CFMLoss, 8-batch average,
batch 128, seed 42, Kaggle P100):

| Model | Params | vs baseline | Eval @ 2 K | Gap to baseline |
|---|---:|---:|---:|---:|
| Baseline (attention + GELU + LayerNorm) | 4,886,544 | 1.00× | **0.1734** | 0.0000 |
| **PhotonFlow v17 champion** ⭐ | **4,934,928** | **1.01×** | **0.2225** | **0.0491** |
| (previous 18.5 M v11 winner) | 18,456,800 | 3.78× | 0.2230 | 0.0496 |

3.75× compression vs v11 with **better** gap.

**v17 winning recipe** (read from `configs/exp2_mnist.yaml`):

* Architecture
  * `hidden_dim=784, num_blocks=14, time_dim=256`
  * `monarch_init='random'` (off-identity Xavier 0.1 — escapes Hardt-Ma identity-near minimum)
  * `SaturableAbsorber(alpha=0.8, leaky_slope=0.05)` (5 % linear bypass past tanh saturation)
  * `adaln_bottleneck=8` (factor `Linear(time_dim, 6·hidden_dim)` as `time_dim → 8 → 6·hidden_dim`; cuts 87 % of per-block params)
  * `gate_init=0.5, adaln_init_std=0.02`
* Training
  * `lr=1.7e-3, warmup=600, cosine to 0 over 2000 steps, batch=128, Adam, seed=42`
  * **plain uniform-t `CFMLoss()`** (no logit-normal / direction / time-weighted loss)
  * `grad_clip=1.0`

For long-horizon training (e.g. 20 K – 200 K steps), set `total_steps` higher
in the YAML and keep the `warmup_steps / total_steps` ratio at ~3 %; lower
`lr` to ~3 e-4 if the run becomes unstable.

References enumerated in `reference.md` and `photonflow_experiments_report.md` §7.


In [ ]:
# -- 1. Environment detection + dependency install -------------------------
import sys, subprocess, os

IN_COLAB  = False
IN_KAGGLE = False
IN_LOCAL  = False

# Check Kaggle FIRST -- Kaggle can also import google.colab.
if os.path.exists('/kaggle/input') or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    IN_KAGGLE = True
    ENV_NAME  = "Kaggle"
else:
    try:
        import google.colab  # noqa: F401
        IN_COLAB = True
        ENV_NAME = "Colab"
    except ImportError:
        IN_LOCAL = True
        ENV_NAME = "Local"

print(f"Environment: {ENV_NAME}")

def _pip_install(*pkgs):
    """Install packages one by one; warn rather than crash."""
    for pkg in pkgs:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                stdout=subprocess.DEVNULL,
                stderr=subprocess.DEVNULL,
            )
            print(f"  [ok] {pkg}")
        except subprocess.CalledProcessError:
            print(f"  [warn] {pkg} -- install failed (may need Internet)")
            if IN_KAGGLE:
                print("         Kaggle: Settings > Internet > toggle ON")

if IN_COLAB or IN_KAGGLE:
    print("Installing dependencies...")
    _pip_install("tqdm", "pyyaml", "wandb")

# --- IMPORTANT: Kaggle's pre-installed torch (cu128) drops sm_60 kernels,
# which fails on Tesla P100 (compute 6.0).  If we got assigned a P100,
# install torch 2.5.1+cu121 which keeps sm_50/60/70/75/80/86/89/90.
if IN_KAGGLE:
    try:
        import torch as _torch_probe
        cap = _torch_probe.cuda.get_device_capability(0)
        if cap[0] < 7:
            print(f"[setup] GPU compute {cap[0]}.{cap[1]} < 7.0 -- reinstalling torch 2.5.1+cu121 for sm_60 support...")
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--index-url", "https://download.pytorch.org/whl/cu121",
                "torch==2.5.1", "torchvision==0.20.1",
            ])
            print("[setup] torch 2.5.1+cu121 installed -- you may need to RESTART the kernel and re-run all cells")
        del _torch_probe
    except Exception as e:
        print(f"[setup] could not probe GPU/torch: {e}")

import torch
assert torch.cuda.is_available(), (
    "GPU not found.\n"
    "  Colab : Runtime > Change runtime type > T4 GPU\n"
    "  Kaggle: Settings > Accelerator > GPU T4 x2 (or P100)"
)
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")


In [ ]:
# -- 2. Repo / sys.path setup ----------------------------------------------
REPO_URL    = "https://github.com/HasinthakaPiyumal/photon-flow-research.git"
REPO_BRANCH = "h/phase1"

if IN_COLAB:
    REPO_DIR = "/content/photon-flow-research"
elif IN_KAGGLE:
    REPO_DIR = "/kaggle/working/photon-flow-research"
else:
    REPO_DIR = ".."  # Notebook lives in notebooks/, repo root is one up

if (IN_COLAB or IN_KAGGLE) and not os.path.exists(REPO_DIR):
    subprocess.check_call([
        "git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR
    ])
    print(f"Cloned repo (branch={REPO_BRANCH}) to {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Repo  : {os.path.abspath(REPO_DIR)}")
print(f"Branch: {REPO_BRANCH}")
print(f"HEAD  : ", end="")
try:
    print(subprocess.check_output(
        ["git", "-C", REPO_DIR, "log", "-1", "--oneline"]
    ).decode().strip())
except Exception:
    print("(not a git checkout)")


In [ ]:
# -- 3. Imports -------------------------------------------------------------
import math, json, time
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# PhotonFlow core (commits 83fda3d + 4f5b08a + 686aa27 add the v17 champion kwargs)
from photonflow.model import PhotonFlowModel       # Dao 2022 Monarch architecture
from photonflow.train import CFMLoss, euler_sample  # Lipman 2023 CFM training

# FID evaluation (Heusel 2017)
try:
    from eval.fid import FIDCalculator
    FID_AVAILABLE = True
except Exception as _e:
    FID_AVAILABLE = False
    print(f"[warn] FID not available ({_e}); cell 11 will skip FID")

device = torch.device("cuda")
print(f"Device: {device}")
print("Imports OK")


In [ ]:
# -- W&B init (get API key from secrets) -----------------------------------
import wandb

# Retrieve WANDB_API_KEY from environment secrets:
#  Colab : Add-ons > Secrets > "WANDB_API_KEY"
#  Kaggle: Add-ons > Secrets > "WANDB_API_KEY" (toggle Internet ON)
#  Local : export WANDB_API_KEY=... in your shell
WANDB_API_KEY = None

if IN_COLAB:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get("WANDB_API_KEY")
    except Exception as e:
        print(f"[warn] Could not load WANDB_API_KEY from Colab secrets: {e}")
elif IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
    except Exception as e:
        print(f"[warn] Could not load WANDB_API_KEY from Kaggle secrets: {e}")
else:
    WANDB_API_KEY = os.environ.get("WANDB_API_KEY")

if WANDB_API_KEY:
    wandb.login(key=WANDB_API_KEY)
    print("W&B login: OK")
else:
    print("[warn] WANDB_API_KEY not found -- running wandb in offline mode")
    os.environ["WANDB_MODE"] = "offline"

wandb_run = None  # initialised in cell 5 once CFG is built
print("W&B cell loaded -- run will be initialised after config is ready")


In [ ]:
# -- 4. Experiment config + W&B run init -----------------------------------
import yaml

config_path = os.path.join(REPO_DIR, "configs", "exp2_mnist.yaml")
with open(config_path) as f:
    yaml_cfg = yaml.safe_load(f)

print(f"Loaded config: {config_path}")

# Pull every v17 champion knob from the YAML.  Anything missing in the YAML
# falls back to the v17-champion default (so older YAMLs still work).
m = yaml_cfg["model"]
t = yaml_cfg["training"]
d = yaml_cfg["data"]

CFG = {
    # ---------- model ----------
    "in_dim":                 m.get("in_dim", 784),
    "hidden_dim":             m.get("hidden_dim", 784),
    "num_blocks":             m.get("num_blocks", 14),
    "time_dim":               m.get("time_dim", 256),
    "use_noise":              m.get("use_noise", False),
    "sigma_s":                m.get("sigma_s", 0.0),
    "sigma_t":                m.get("sigma_t", 0.0),
    "gate_init":              m.get("gate_init", 0.5),
    "adaln_init_std":         m.get("adaln_init_std", 0.02),
    "adaln_bottleneck":       m.get("adaln_bottleneck", 8),
    "num_monarch_factors":    m.get("num_monarch_factors", 1),
    "monarch_init":           m.get("monarch_init", "random"),
    "depth_decay_residual":   m.get("depth_decay_residual", False),
    "absorber_alpha":         m.get("absorber_alpha", 0.8),
    "absorber_leaky_slope":   m.get("absorber_leaky_slope", 0.05),
    "learnable_absorber_alpha": m.get("learnable_absorber_alpha", False),
    "mean_center_norm":       m.get("mean_center_norm", False),
    "seq_dim":                m.get("seq_dim", None),
    "feat_dim":               m.get("feat_dim", None),
    # ---------- data ----------
    "dataset":                d["dataset"],
    "batch_size":             d.get("batch_size", 128),
    "data_root":              os.path.join(REPO_DIR, d.get("root", "./data")),
    "num_workers":            d.get("num_workers", 2),
    # ---------- training ----------
    "lr":                     t.get("lr", 1.7e-3),
    "total_steps":            t.get("total_steps", 2000),
    "warmup_steps":           t.get("warmup_steps", 600),
    "lr_schedule":            t.get("lr_schedule", "cosine"),
    "grad_clip":              t.get("grad_clip", 1.0),
    "checkpoint_every":       t.get("checkpoint_every", 1000),
    "sample_every":           t.get("sample_every", 500),
    "sample_steps":           t.get("sample_steps", 20),
    "seed":                   t.get("seed", 42),
    "time_sampling":          t.get("time_sampling", "uniform"),
    "direction_loss_weight":  t.get("direction_loss_weight", 0.0),
    "loss_weight_gamma":      t.get("loss_weight_gamma", 0.0),
    "logit_normal_mean":      t.get("logit_normal_mean", 0.0),
    "logit_normal_std":       t.get("logit_normal_std", 1.0),
    # ---------- io ----------
    "output_dir":             os.path.join(REPO_DIR, yaml_cfg.get("output_dir", "outputs/exp2_photonflow_mnist")),
}

# Windows-multiprocessing safety: Lambda transforms can't be pickled across
# DataLoader workers on Windows.  Force num_workers=0 on Windows.
if IN_LOCAL and sys.platform.startswith("win"):
    CFG["num_workers"] = 0

IN_DIM = CFG["in_dim"]

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

OUTPUT_DIR = CFG["output_dir"]
CKPT_DIR   = os.path.join(OUTPUT_DIR, "checkpoints")
FIG_DIR    = os.path.join(OUTPUT_DIR, "figures")
for _d in [CKPT_DIR, FIG_DIR]:
    os.makedirs(_d, exist_ok=True)

_ts_k = max(1, CFG["total_steps"] // 1000)
wandb_run = wandb.init(
    project="photonflow",
    name=f"exp2-photonflow-v17-{_ts_k}K-{ENV_NAME.lower()}",
    config=CFG,
    tags=["exp2", "photonflow", "monarch", "v17-champion", "mnist", ENV_NAME.lower()],
    notes=yaml_cfg["experiment"]["description"],
    reinit=True,
)

# Pretty-print the resolved CFG (only the most important fields).
print()
print("=" * 60)
print("PhotonFlow v17 champion config")
print("=" * 60)
print(f"  hidden_dim={CFG['hidden_dim']}  num_blocks={CFG['num_blocks']}  time_dim={CFG['time_dim']}")
print(f"  monarch_init={CFG['monarch_init']!r}  num_monarch_factors={CFG['num_monarch_factors']}")
print(f"  gate_init={CFG['gate_init']}  adaln_init_std={CFG['adaln_init_std']}")
print(f"  adaln_bottleneck={CFG['adaln_bottleneck']}  depth_decay_residual={CFG['depth_decay_residual']}")
print(f"  absorber_alpha={CFG['absorber_alpha']}  leaky_slope={CFG['absorber_leaky_slope']}")
print(f"  use_noise={CFG['use_noise']}  mean_center_norm={CFG['mean_center_norm']}")
print()
print(f"  lr={CFG['lr']}  warmup={CFG['warmup_steps']}/{CFG['total_steps']}  schedule={CFG['lr_schedule']}")
print(f"  batch_size={CFG['batch_size']}  num_workers={CFG['num_workers']}  seed={CFG['seed']}")
print(f"  loss=CFMLoss(time_sampling={CFG['time_sampling']!r}, direction={CFG['direction_loss_weight']}, gamma={CFG['loss_weight_gamma']})")
print()
print(f"  W&B run -> {wandb_run.url}")


In [ ]:
# -- 5. MNIST DataLoader ---------------------------------------------------
# Note: flatten (1,28,28) -> (784,) is done INLINE in the loop, NOT via
# transforms.Lambda, because Windows multiprocessing DataLoaders can't pickle
# local lambdas.  Semantically identical to the baseline notebook's pipeline.
_tfm = transforms.ToTensor()

train_ds = datasets.MNIST(CFG["data_root"], train=True,  download=True, transform=_tfm)
test_ds  = datasets.MNIST(CFG["data_root"], train=False, download=True, transform=_tfm)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True, num_workers=CFG["num_workers"],
    pin_memory=True, drop_last=True,
)

# Separate eval loader for the uniform-t baseline-comparable eval loss.
# Deterministic seed so eval values are repeatable across training steps.
eval_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True, num_workers=0, pin_memory=True, drop_last=True,
    generator=torch.Generator().manual_seed(CFG["seed"] + 1),
)

steps_per_epoch = len(train_loader)
total_epochs    = CFG["total_steps"] / steps_per_epoch
print(f"Train: {len(train_ds):,}  Test: {len(test_ds):,}")
print(f"Batch size: {CFG['batch_size']}  Steps/epoch: {steps_per_epoch}")
print(f"Total epochs: {total_epochs:.1f}  ({CFG['total_steps']:,} steps)")


In [ ]:
# -- 6. PhotonFlowModel (v17 champion) + LR scheduler ---------------------
def _build_model():
    return PhotonFlowModel(
        in_dim                    = CFG["in_dim"],
        hidden_dim                = CFG["hidden_dim"],
        num_blocks                = CFG["num_blocks"],
        time_dim                  = CFG["time_dim"],
        use_noise                 = CFG["use_noise"],
        sigma_s                   = CFG["sigma_s"],
        sigma_t                   = CFG["sigma_t"],
        gate_init                 = CFG["gate_init"],
        adaln_init_std            = CFG["adaln_init_std"],
        adaln_bottleneck          = CFG["adaln_bottleneck"],
        num_monarch_factors       = CFG["num_monarch_factors"],
        monarch_init              = CFG["monarch_init"],
        depth_decay_residual      = CFG["depth_decay_residual"],
        absorber_alpha            = CFG["absorber_alpha"],
        absorber_leaky_slope      = CFG["absorber_leaky_slope"],
        learnable_absorber_alpha  = CFG["learnable_absorber_alpha"],
        mean_center_norm          = CFG["mean_center_norm"],
        seq_dim                   = CFG["seq_dim"],
        feat_dim                  = CFG["feat_dim"],
    ).to(device)

def _build_lr_scheduler(opt, total_steps, warmup_steps, schedule):
    """Linear warmup then cosine decay to 0; or constant if schedule != cosine."""
    from torch.optim.lr_scheduler import LambdaLR
    if schedule == "cosine":
        def _fn(step):
            if step < warmup_steps:
                return step / max(1, warmup_steps)
            p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
            return 0.5 * (1.0 + math.cos(math.pi * p))
        return LambdaLR(opt, _fn)
    return None

model     = _build_model()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
scheduler = _build_lr_scheduler(optimizer, CFG["total_steps"], CFG["warmup_steps"], CFG["lr_schedule"])

# Two CFMLoss instances:
#   loss_fn      -- training (whatever's in CFG; defaults to plain uniform-t)
#   eval_loss_fn -- always plain uniform-t, for fair comparison with the baseline.
loss_fn = CFMLoss(
    sigma_min=0.0,
    time_sampling          = CFG["time_sampling"],
    logit_normal_mean      = CFG["logit_normal_mean"],
    logit_normal_std       = CFG["logit_normal_std"],
    direction_loss_weight  = CFG["direction_loss_weight"],
    loss_weight_gamma      = CFG["loss_weight_gamma"],
)
eval_loss_fn = CFMLoss()  # uniform-t plain MSE

total_params = model.count_parameters()
print(f"Model: PhotonFlowModel (v17 champion)")
print(f"  params = {total_params:,}")
print(f"  baseline reference for comparison = 4,886,544 params (DiT/attention 4-block)")
ratio = total_params / 4_886_544
print(f"  -> {ratio:.2f}x baseline")
wandb.log({"model/total_params": total_params, "model/params_vs_baseline": ratio})

# Sanity: forward pass + backward, no NaN, all params receive gradient.
with torch.no_grad():
    _x = torch.randn(4, CFG["in_dim"], device=device)
    _t = torch.rand(4, device=device)
    _y = model(_x, _t)
    _max = _y.abs().max().item()
print(f"Init forward OK: out shape {_y.shape}, max abs {_max:.4f}")

_loss = loss_fn(model, _x.detach())
_loss.backward()
_no_grad = [n for n, p in model.named_parameters() if p.grad is None]
assert not _no_grad, f"Params with no gradient: {_no_grad}"
optimizer.zero_grad(set_to_none=True)
print(f"Init backward OK: loss = {_loss.item():.4f}, all {sum(1 for _ in model.parameters())} params have grad")


In [ ]:
# -- 7. VERIFY FIRST 500 STEPS (quick sanity check) -----------------------
VERIFY_STEPS = min(500, CFG["total_steps"] // 4)

print(f"Verification run: {VERIFY_STEPS} steps ...")
verify_losses = []
data_iter_v = iter(train_loader)
model.train()

for step in range(VERIFY_STEPS):
    try: x1, _ = next(data_iter_v)
    except StopIteration:
        data_iter_v = iter(train_loader); x1, _ = next(data_iter_v)
    x1 = x1.view(x1.size(0), -1).to(device, non_blocking=True)
    loss = loss_fn(model, x1)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    if CFG["grad_clip"] > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
    optimizer.step()
    if scheduler is not None:
        scheduler.step()
    verify_losses.append(loss.item())
    if (step + 1) % 100 == 0:
        avg = float(np.mean(verify_losses[-100:]))
        print(f"  step {step+1:>4d}/{VERIFY_STEPS}  loss={avg:.4f}")

first_100 = float(np.mean(verify_losses[:100]))
last_100  = float(np.mean(verify_losses[-100:]))
any_nan   = any(math.isnan(l) for l in verify_losses)
decreased = last_100 < first_100

print()
print(f"  Loss first 100: {first_100:.4f}")
print(f"  Loss last  100: {last_100:.4f}")
print(f"  Decreased:      {decreased}")
print(f"  Any NaN:        {any_nan}")

# Quick sample grid (16 images)
model.eval()
with torch.no_grad():
    quick_samp = euler_sample(model, shape=(16, IN_DIM),
                              num_steps=CFG["sample_steps"], device=str(device))
quick_samp = quick_samp.clamp(0.0, 1.0).cpu().view(16, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    ax.imshow(quick_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(f"Exp2 verification ({VERIFY_STEPS} steps, loss={last_100:.4f})", fontsize=11)
plt.tight_layout()
wandb.log({"verify/samples": wandb.Image(fig)})
plt.show(); plt.close(fig)

if decreased and not any_nan:
    print("\n" + "=" * 50)
    print("  VERDICT: GO -- proceed to full training (Cell 8)")
    print("=" * 50)
else:
    print("\n" + "=" * 50)
    print("  VERDICT: NO-GO -- debug before full training")
    print("=" * 50)

# Reset for the real run.  Reseed and rebuild model+opt+sched so the verify
# steps don't bias the full training trajectory.
torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])
model     = _build_model()
optimizer = torch.optim.Adam(model.parameters(), lr=CFG["lr"])
scheduler = _build_lr_scheduler(optimizer, CFG["total_steps"], CFG["warmup_steps"], CFG["lr_schedule"])
print(f"\nModel reset for full {CFG['total_steps']:,}-step training.")


In [ ]:
# -- 8. Training loop -----------------------------------------------------
# Logs:
#   train/loss              -- per-step training loss (whatever loss_fn is)
#   train/loss_avg100       -- 100-step rolling average
#   eval/uniform_t_loss     -- plain-CFM uniform-t eval (every CFG['sample_every']
#                              steps), 8-batch average -- DIRECTLY COMPARABLE
#                              with baseline notebook's reported loss
#   gate_mean/block_i       -- mean of the per-dim gate biases for block i
#   lr/value                -- current optimizer learning rate
@torch.no_grad()
def _compute_eval_loss(n_batches=8):
    """Uniform-t, plain-MSE CFM loss averaged over n_batches eval batches."""
    model.eval()
    it = iter(eval_loader)
    tot, n = 0.0, 0
    for _ in range(n_batches):
        try: xe, _ = next(it)
        except StopIteration: break
        xe = xe.view(xe.size(0), -1).to(device, non_blocking=True)
        tot += eval_loss_fn(model, xe).item()
        n += 1
    model.train()
    return tot / max(1, n)

losses        = []   # per-step training loss
step_log      = []   # (step, avg100) for the loss-curve plot
eval_log      = []   # (step, eval_loss) every sample_every steps
data_iter     = iter(train_loader)
model.train()
t_start       = time.time()
best_eval     = float("inf")

pbar = tqdm(range(CFG["total_steps"]), desc="exp2 PhotonFlow v17", dynamic_ncols=True)

for step in pbar:
    try:
        x1, _ = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        x1, _ = next(data_iter)
    x1 = x1.view(x1.size(0), -1).to(device, non_blocking=True)

    loss = loss_fn(model, x1, step=step, total_steps=CFG["total_steps"])
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    if CFG["grad_clip"] > 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
    optimizer.step()
    if scheduler is not None:
        scheduler.step()

    losses.append(loss.item())
    wandb.log({"train/loss": loss.item(), "lr/value": optimizer.param_groups[0]["lr"]}, step=step)

    if step % 100 == 0:
        avg = float(np.mean(losses[-100:]))
        step_log.append((step, avg))
        pbar.set_postfix(loss=f"{avg:.4f}")
        wandb.log({"train/loss_avg100": avg}, step=step)

    # gate diagnostics every 10 % of the run (or every 1K, whichever is larger)
    diag_every = max(1000, CFG["total_steps"] // 10)
    if step % diag_every == 0:
        gates = {}
        for i, blk in enumerate(model.blocks):
            # adaLN_proj last layer's biases, slot 2 (gate1) and 5 (gate2)
            b = blk.adaLN_proj[-1].bias
            d = blk.dim
            gates[f"gate1_mean/block_{i}"] = b[2*d:3*d].mean().item()
            gates[f"gate2_mean/block_{i}"] = b[5*d:6*d].mean().item()
        wandb.log(gates, step=step)

    # uniform-t eval loss (baseline-comparable) + sample grid every sample_every
    if (step + 1) % CFG["sample_every"] == 0:
        ev = _compute_eval_loss(n_batches=8)
        eval_log.append((step + 1, ev))
        if ev < best_eval: best_eval = ev
        wandb.log({"eval/uniform_t_loss": ev, "eval/best": best_eval}, step=step)
        elapsed = time.time() - t_start
        print(f"  [step {step+1:>5d}] eval={ev:.4f}  best={best_eval:.4f}  elapsed={elapsed:.0f}s")

        # Sample grid
        model.eval()
        with torch.no_grad():
            samp = euler_sample(model, shape=(64, IN_DIM),
                                num_steps=CFG["sample_steps"], device=str(device))
        samp = samp.clamp(0.0, 1.0).cpu().view(64, 1, 28, 28)
        fig, axes = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(axes.flat):
            ax.imshow(samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
            ax.axis("off")
        fig.suptitle(f"Exp2 v17  step={step+1:,}  eval={ev:.4f}", fontsize=11)
        plt.tight_layout()
        fig_path = os.path.join(FIG_DIR, f"samples_step{step+1:07d}.png")
        plt.savefig(fig_path, dpi=100, bbox_inches="tight")
        wandb.log({f"samples/step_{step+1}": wandb.Image(fig)}, step=step)
        plt.show(); plt.close(fig)
        model.train()

    # Checkpoint
    if (step + 1) % CFG["checkpoint_every"] == 0:
        ckpt_path = os.path.join(CKPT_DIR, f"step_{step+1:07d}.pt")
        torch.save({
            "step": step + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "losses": losses,
            "config": CFG,
        }, ckpt_path)

pbar.close()
total_time = time.time() - t_start
print(f"\nTraining complete: {len(losses):,} steps in {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  final train loss (avg of last 100): {np.mean(losses[-100:]):.4f}")
print(f"  best uniform-t eval loss:           {best_eval:.4f}")
# Compare to the reference baseline at 2K steps (fixed reference from the sweep).
BASELINE_2K_REF = 0.1734
gap = best_eval - BASELINE_2K_REF
print(f"  baseline 2K reference eval:         {BASELINE_2K_REF:.4f}")
print(f"  gap-to-baseline (best_eval - ref):  {gap:+.4f}")
wandb.log({"summary/baseline_ref": BASELINE_2K_REF, "summary/gap_vs_baseline": gap})


In [ ]:
# -- 9. Loss curves (training + uniform-t eval) ---------------------------
fig, ax = plt.subplots(figsize=(11, 4.5))

# Training loss (avg100), in steps
if step_log:
    s, l = zip(*step_log)
    ax.plot(s, l, lw=1.4, alpha=0.85, color="tab:orange",
            label="train loss (avg of 100 steps)")

# Eval loss (uniform-t), as scatter points joined by a line
if eval_log:
    es, el = zip(*eval_log)
    ax.plot(es, el, "-o", lw=1.6, ms=4.5, color="tab:blue",
            label="eval loss (uniform-t, 8-batch avg)")

# Reference line for baseline at 2K
ax.axhline(0.1734, ls="--", color="gray", alpha=0.6, label="baseline @ 2K (0.1734)")
ax.axhline(0.1734 + 0.05, ls=":", color="green", alpha=0.5, label="target gap = 0.05")

ax.set_xlabel("Training step", fontsize=12)
ax.set_ylabel("CFM loss (MSE)", fontsize=12)
_ts_k = max(1, CFG["total_steps"] // 1000)
ax.set_title(f"Exp2 PhotonFlow v17 -- {_ts_k}K steps  ({total_params:,} params, {total_params/4886544:.2f}x baseline)", fontsize=12)
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
curve_path = os.path.join(FIG_DIR, "loss_curve.png")
plt.savefig(curve_path, dpi=150)
wandb.log({"charts/loss_curve": wandb.Image(fig)})
plt.show(); plt.close(fig)
print(f"Loss curve saved: {curve_path}")


In [ ]:
# -- 10. Final 10x10 sample grid ------------------------------------------
model.eval()
with torch.no_grad():
    final_samp = euler_sample(model, shape=(100, IN_DIM),
                              num_steps=CFG["sample_steps"], device=str(device))
final_samp = final_samp.clamp(0.0, 1.0).cpu().view(100, 1, 28, 28)

fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(final_samp[i, 0].numpy(), cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle(f"Exp2 v17 final samples  (step {CFG['total_steps']:,})", fontsize=13)
plt.tight_layout()
samp_path = os.path.join(FIG_DIR, "final_samples.png")
plt.savefig(samp_path, dpi=150, bbox_inches="tight")
wandb.log({"samples/final_10x10": wandb.Image(fig)})
plt.show(); plt.close(fig)
print(f"Final samples saved: {samp_path}")


In [ ]:
# -- 11. FID computation (optional) ---------------------------------------
fid_score = float("nan")
if not FID_AVAILABLE:
    print("[skip] eval.fid not importable; skipping FID")
else:
    N_FID    = 10_000
    fid_calc = FIDCalculator(device=device)

    real_loader = DataLoader(test_ds, batch_size=256, shuffle=False, num_workers=0)
    real_imgs = []
    for imgs, _ in real_loader:
        real_imgs.append(imgs.view(-1, 1, 28, 28))
    real_imgs = torch.cat(real_imgs, dim=0)[:N_FID]
    print(f"Real images: {real_imgs.shape}")

    model.eval()
    gen_batches = []
    GEN_BATCH   = 256
    n_generated = 0
    with torch.no_grad():
        pbar_fid = tqdm(total=N_FID, desc="Generating for FID", unit="img")
        while n_generated < N_FID:
            bs   = min(GEN_BATCH, N_FID - n_generated)
            samp = euler_sample(model, shape=(bs, IN_DIM),
                                num_steps=CFG["sample_steps"], device=str(device))
            gen_batches.append(samp.clamp(0.0, 1.0).cpu().view(bs, 1, 28, 28))
            n_generated += bs
            pbar_fid.update(bs)
        pbar_fid.close()
    gen_imgs = torch.cat(gen_batches, dim=0)[:N_FID]

    print("Extracting features...")
    real_feats = fid_calc.extract_features(real_imgs, batch_size=256)
    gen_feats  = fid_calc.extract_features(gen_imgs,  batch_size=256)
    real_stats = fid_calc.compute_statistics(real_feats)
    gen_stats  = fid_calc.compute_statistics(gen_feats)
    fid_score  = fid_calc.compute_fid(real_stats, gen_stats)
    _ts_k = max(1, CFG["total_steps"] // 1000)
    print(f"\nFID (PhotonFlow v17, {_ts_k}K steps): {fid_score:.2f}")
    wandb.log({"eval/fid": fid_score})

    # Compare to exp1 baseline if its results.json exists
    exp1_path = os.path.join(REPO_DIR, "outputs", "exp1_baseline", "results_exp1.json")
    if os.path.exists(exp1_path):
        with open(exp1_path) as f:
            exp1 = json.load(f)
        exp1_fid  = exp1.get("fid", float("nan"))
        delta_pct = ((fid_score - exp1_fid) / exp1_fid) * 100 if exp1_fid > 0 else float("nan")
        target    = exp1_fid * 1.10
        passed    = fid_score <= target
        print(f"Exp1 baseline FID: {exp1_fid:.2f}")
        print(f"FID delta:         {delta_pct:+.1f}%   (target: within +10%)")
        print(f"Verdict:           {'PASS' if passed else 'FAIL'}  (target FID <= {target:.2f})")
        wandb.log({"eval/exp1_fid": exp1_fid, "eval/fid_delta_pct": delta_pct})
    else:
        print("(exp1 results not found -- run notebook 02 first)")


In [ ]:
# -- 12. Results summary + W&B finish -------------------------------------
results = {
    "experiment":  "exp2_photonflow_mnist_v17",
    "environment": ENV_NAME,
    "description": "PhotonFlow v17 champion (slimmed Monarch + leaky absorber + adaLN bottleneck)",
    "total_steps": CFG["total_steps"],
    "fid":         round(float(fid_score), 4) if not math.isnan(fid_score) else None,
    "final_loss_avg100":          round(float(np.mean(losses[-100:])), 6),
    "best_uniform_t_eval_loss":   round(float(best_eval), 6),
    "baseline_2k_reference_eval": 0.1734,
    "gap_vs_baseline":            round(float(best_eval) - 0.1734, 6),
    "model_params":               total_params,
    "params_vs_baseline":         round(total_params / 4_886_544, 4),
    "architecture": {
        "type":                       "photonflow",
        "hidden_dim":                 CFG["hidden_dim"],
        "num_blocks":                 CFG["num_blocks"],
        "time_dim":                   CFG["time_dim"],
        "monarch_init":               CFG["monarch_init"],
        "num_monarch_factors":        CFG["num_monarch_factors"],
        "gate_init":                  CFG["gate_init"],
        "adaln_init_std":             CFG["adaln_init_std"],
        "adaln_bottleneck":           CFG["adaln_bottleneck"],
        "absorber_alpha":             CFG["absorber_alpha"],
        "absorber_leaky_slope":       CFG["absorber_leaky_slope"],
        "depth_decay_residual":       CFG["depth_decay_residual"],
        "use_noise":                  CFG["use_noise"],
    },
    "training": {
        "optimizer":             "Adam",
        "lr":                    CFG["lr"],
        "warmup_steps":          CFG["warmup_steps"],
        "lr_schedule":           CFG["lr_schedule"],
        "grad_clip":             CFG["grad_clip"],
        "batch_size":            CFG["batch_size"],
        "dataset":               CFG["dataset"],
        "seed":                  CFG["seed"],
        "loss":                  "CFMLoss(uniform-t, plain MSE)",
    },
    "sources": [
        "Dao et al. 2022 -- Monarch matrices M=PLP^TR (ICML, Def 3.1)",
        "Lipman et al. 2023 -- Flow Matching CFM loss (ICLR, Eq. 23)",
        "Peebles & Xie 2023 -- adaLN-Zero conditioning (DiT, ICCV)",
        "Shen et al. 2017 -- Saturable absorber on MZI mesh (Nature Photonics)",
        "Hardt & Ma 2017 -- identity-near minima in residual networks (arXiv:1611.04231)",
        "Wang et al. 2023 -- Monarch Mixer (NeurIPS, arXiv:2310.12109)",
        "Internal report -- photonflow_experiments_report.md (full v1-v17 sweep)",
    ],
}

results_path = os.path.join(OUTPUT_DIR, "results_exp2.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

np.save(os.path.join(OUTPUT_DIR, "losses_exp2.npy"), np.array(losses))
if eval_log:
    np.save(os.path.join(OUTPUT_DIR, "eval_log_exp2.npy"), np.array(eval_log))

final_ckpt = os.path.join(CKPT_DIR, "exp2_final.pt")
torch.save({
    "step":             CFG["total_steps"],
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "results":          results,
    "config":           CFG,
}, final_ckpt)

wandb.log({
    "summary/fid":               fid_score if not math.isnan(fid_score) else 0.0,
    "summary/final_train_loss":  float(np.mean(losses[-100:])),
    "summary/best_eval":         best_eval,
    "summary/gap_vs_baseline":   best_eval - 0.1734,
    "summary/total_params":      total_params,
})
wandb.save(results_path)
wandb.finish()

SEP = "=" * 64
print(SEP)
print("EXP2 PHOTONFLOW v17 RESULTS")
print(SEP)
print(f"  Model parameters:         {total_params:,}  ({total_params/4886544:.2f}x baseline)")
print(f"  Training steps:           {CFG['total_steps']:,}")
print(f"  Final train loss (100):   {float(np.mean(losses[-100:])):.4f}")
print(f"  Best uniform-t eval:      {best_eval:.4f}")
print(f"  Baseline 2K reference:    0.1734")
print(f"  Gap vs baseline:          {best_eval - 0.1734:+.4f}")
if not math.isnan(fid_score):
    print(f"  FID (10K samples):        {fid_score:.2f}")
print(f"  Environment:              {ENV_NAME}")
print(SEP)
print(f"  Results    -> {results_path}")
print(f"  Checkpoint -> {final_ckpt}")
print(f"  W&B run    -> {wandb_run.url}")
print(SEP)
print()
print("Next: notebooks/04_exp3_noise_regularized.ipynb")
print("      Add shot noise (sigma_s=0.02) + thermal crosstalk (sigma_t=0.01)")
